In [ ]:
!pip install -q transformers accelerate peft bitsandbytes trl datasets

In [ ]:
# !pip uninstall -y pyarrow datasets trl
# !pip install -U pyarrow datasets trl

In [ ]:
import os
import json
import ast
import time
import torch
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from collections import Counter
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATASET_PATH = "/content/drive/MyDrive/term_paper_masha/bank_reviews_dataset.csv"

teacher_path = "/content/drive/MyDrive/term_paper_masha/predictions_clean_full_aspects.csv"
manual_path = "/content/drive/MyDrive/term_paper_masha/labeled_reviews_with_additional_with_labels_list.csv"

In [ ]:
reviews_df = pd.read_csv(DATASET_PATH, sep=";", encoding="utf-8-sig")
print("Исходный датасет:", reviews_df.shape)

reviews_df.head()

In [ ]:
teacher_df = pd.read_csv(teacher_path, sep=";", encoding="utf-8-sig")
print("Teacher predictions:", teacher_df.shape)
teacher_df.head()

In [ ]:
def safe_parse(value):
    if pd.isna(value):
        return None
    if isinstance(value, (dict, list)):
        return value
    value = str(value).strip()
    if value == "":
        return None
    try:
        return json.loads(value)
    except Exception:
        pass
    try:
        return ast.literal_eval(value)
    except Exception:
        return None

In [ ]:
def extract_aspects_from_row(row):
    possible_cols = [
        "prediction",
        "prediction_clean",
        "clean_prediction",
        "aspects",
        "output"
    ]

    parsed = None
    for col in possible_cols:
        if col in row.index:
            parsed = safe_parse(row[col])
            if parsed is not None:
                break
    if parsed is None:
        return []
    if isinstance(parsed, dict):
        aspects = parsed.get("aspects", [])
    elif isinstance(parsed, list):
        aspects = parsed
    else:
        aspects = []
    clean_aspects = []
    for item in aspects:
        if not isinstance(item, dict):
            continue
        label = item.get("label")
        evidence = item.get("evidence", "")
        if label:
            clean_aspects.append({"label": label, "evidence": evidence})

    return clean_aspects

In [ ]:
train_rows = []
for _, row in teacher_df.iterrows():
    aspects = extract_aspects_from_row(row)
    train_rows.append({"id": row["id"], "text": row["text"], "output": {"aspects": aspects}})

train_df_full = pd.DataFrame(train_rows)
print("Prepared teacher dataset:", train_df_full.shape)
train_df_full.head()

In [ ]:
train_df, test_df = train_test_split(
    train_df_full,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
print("Train:", train_df.shape)
print("Test:", test_df.shape)

In [ ]:
PROMPT = """
/no_think
Ты размечаешь отзывы клиентов банка для задачи Aspect-Based Sentiment Analysis.

Нужно найти в отзыве фрагменты текста, которые относятся к банковским аспектам,
и определить тональность каждого аспекта. Не рассуждай. Не объясняй.

Разрешенные labels:

CARDS_POS / CARDS_NEG — карты, выпуск, доставка, блокировка, пин-код, кэшбек, бонусы по карте.
DIGITAL_POS / DIGITAL_NEG — мобильное приложение, интернет-банк, личный кабинет, вход, авторизация.
SUPPORT_POS / SUPPORT_NEG — поддержка, чат, горячая линия, оператор, контактный центр.
PAYMENTS_POS / PAYMENTS_NEG — платежи, переводы, СБП, оплата, зачисление, списание.
CREDITS_POS / CREDITS_NEG — кредиты, ипотека, автокредит, график платежей, досрочное погашение.
FEES_POS / FEES_NEG — комиссии, платное обслуживание, скрытые списания, тарифы.
OFFICE_POS / OFFICE_NEG — офис, отделение, касса, сотрудники в офисе, очередь.
ATM_POS / ATM_NEG — банкоматы, снятие наличных, внесение наличных.
FRAUD_POS / FRAUD_NEG — мошенничество, подозрительные операции, безопасность, 115-ФЗ, блокировки из-за финмониторинга.
INSURANCE_POS / INSURANCE_NEG — страховки, страхование, финансовая защита.
INVESTMENTS_POS / INVESTMENTS_NEG — инвестиции, брокерский счет, ИИС, ПИФ.
PREMIUM_POS / PREMIUM_NEG — премиальное обслуживание, персональный менеджер, private banking.
OTHER_POS / OTHER_NEG — важный банковский аспект, который не попал в категории выше.

Правила:
1. Верни только JSON, без пояснений. Не рассуждай. Не объясняй.
2. Не придумывай аспекты, которых нет в тексте.
3. evidence должен быть точной цитатой из отзыва.
4. Один отзыв может содержать несколько аспектов.
5. Если аспектов нет, верни {{"aspects": []}}.
6. Sentiment определяй по отношению клиента к конкретному аспекту.

Формат ответа:
{{
  "aspects": [
    {{
      "label": "CARDS_NEG",
      "evidence": "заблокировали карту"
    }}
  ]
}}

Отзыв:
{text}

Верни только JSON.



Твоя задача — найти ВСЕ аспекты в отзыве банка.

Один отзыв может содержать несколько аспектов.

Не останавливайся после первого найденного аспекта.

Разрешенные labels:
DIGITAL_NEG, DIGITAL_POS, SUPPORT_NEG, SUPPORT_POS, CARDS_NEG, CARDS_POS, ATM_NEG, ATM_POS, FRAUD_NEG, FRAUD_POS, OTHER_NEG, OTHER_POS.

Формат:
{{"aspects":[{{"label":"DIGITAL_NEG","evidence":"Мобильное приложение ужасное"}}]}}

Отзыв:
{text}
""".strip()

In [ ]:
MODEL_NAME = "Qwen/Qwen3-1.7B"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def make_training_text(row):
    user_text = str(row["text"])
    output_text = json.dumps(row["output"], ensure_ascii=False)

    messages = [
        {"role": "system", "content": PROMPT},
        {"role": "user", "content": user_text},
        {"role": "assistant", "content": output_text}
    ]

    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

In [ ]:
train_df["training_text"] = train_df.apply(make_training_text, axis=1)
test_df["training_text"] = test_df.apply(make_training_text, axis=1)
train_dataset = Dataset.from_pandas(train_df[["training_text"]])
test_dataset = Dataset.from_pandas(test_df[["training_text"]])

print(train_dataset)
print(test_dataset)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_steps=200,
    save_total_limit=2,
    fp16=True,
    bf16=False,
    optim="paged_adamw_8bit",
    report_to="none",
    remove_unused_columns=False
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    dataset_text_field="training_text",
    max_seq_length=2048,
    args=training_args
)

In [ ]:
trainer.train()

In [ ]:
OUTPUT_DIR = "./qwen3_1_7b_student_lora"

STUDENT_PREDICTIONS_CSV = "./qwen3_1_7b_student_predictions_full.csv"
STUDENT_PREDICTIONS_JSON = "./qwen3_1_7b_student_predictions_full.json"
METRICS_PATH = "./qwen3_1_7b_student_metrics.xlsx"

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Student model saved:", OUTPUT_DIR)

In [ ]:
def extract_json_from_answer(answer):
    answer = str(answer).strip()
    try:
        return json.loads(answer)
    except Exception:
        pass

    start = answer.find("{")
    end = answer.rfind("}")
    if start != -1 and end != -1 and end > start:
        json_part = answer[start:end + 1]
        try:
            return json.loads(json_part)
        except Exception:
            pass

    return {"aspects": []}

In [ ]:
def predict_absa(text, max_new_tokens=512):
    messages = [
        {"role": "system", "content": PROMPT},
        {"role": "user", "content": str(text)}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs,  max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    return extract_json_from_answer(answer)

In [ ]:
generation_df = reviews_df.copy().reset_index(drop=True)

print("Reviews for prediction:", generation_df.shape)
generation_df.head()

In [ ]:
predictions = []

for _, row in tqdm(generation_df.iterrows(), total=len(generation_df)):
    result = predict_absa(str(row["text"]))
    labels_list = [aspect.get("label") for aspect in result.get("aspects", []) if isinstance(aspect, dict) and aspect.get("label")]

    predictions.append({
        "id": int(row["id"]),
        "text": str(row["text"]),
        "prediction": result,
        "labels_list": labels_list
    })

    if len(predictions) % 100 == 0:
        partial_df = pd.DataFrame(predictions).copy()
        partial_df["prediction"] = partial_df["prediction"].apply(lambda x: json.dumps(x, ensure_ascii=False))
        partial_df["labels_list"] = partial_df["labels_list"].apply(lambda x: json.dumps(x, ensure_ascii=False))
        partial_df.to_csv(PARTIAL_PREDICTIONS_CSV, index=False, encoding="utf-8-sig")

        print("Saved partial:", len(predictions))

In [ ]:
student_pred_df = pd.DataFrame(predictions)

student_pred_df["prediction"] = student_pred_df["prediction"].apply(lambda x: json.dumps(x, ensure_ascii=False))
student_pred_df["labels_list"] = student_pred_df["labels_list"].apply(lambda x: json.dumps(x, ensure_ascii=False))
student_pred_df.to_csv(STUDENT_PREDICTIONS_CSV, index=False, encoding="utf-8-sig")

with open(STUDENT_PREDICTIONS_JSON, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

print("Saved CSV:", STUDENT_PREDICTIONS_CSV)
print("Saved JSON:", STUDENT_PREDICTIONS_JSON)

student_pred_df.head()

In [ ]:
def get_labels_list_from_row(row):
    if "labels_list" in row.index:
        parsed = safe_parse(row["labels_list"])
        if isinstance(parsed, list):
            return [str(x) for x in parsed if pd.notna(x)]

    aspects = extract_aspects_from_row(row)

    return [aspect["label"] for aspect in aspects if isinstance(aspect, dict) and aspect.get("label")]

In [ ]:
def prepare_eval_df(path):
    df = pd.read_csv(path, encoding="utf-8-sig")

    df["labels_list"] = df.apply(get_labels_list_from_row, axis=1)

    return df[["id", "labels_list"]].copy()

In [ ]:
def calculate_metrics_by_labels(base_df, pred_df):
    merged = base_df.merge(pred_df, on="id", how="inner", suffixes=("_true", "_pred"))
    tp = 0
    fp = 0
    fn = 0
    exact_matches = []
    soft_scores = []

    for _, row in merged.iterrows():
        true_labels = row["labels_list_true"]
        pred_labels = row["labels_list_pred"]
        true_counter = Counter(true_labels)
        pred_counter = Counter(pred_labels)

        current_tp = sum((true_counter & pred_counter).values())
        current_fp = sum((pred_counter - true_counter).values())
        current_fn = sum((true_counter - pred_counter).values())
        tp += current_tp
        fp += current_fp
        fn += current_fn

        exact_matches.append(true_counter == pred_counter)
        if len(true_labels) == 0 and len(pred_labels) == 0:
            soft_scores.append(1)
        elif len(true_labels) == 0 or len(pred_labels) == 0:
            soft_scores.append(0)
        else:
            soft_scores.append(current_tp / max(len(true_labels), len(pred_labels)))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = (2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0)
    exact_match_accuracy = np.mean(exact_matches) if exact_matches else 0
    soft_accuracy = np.mean(soft_scores) if soft_scores else 0

    return {
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Exact Match Accuracy": exact_match_accuracy,
        "Soft Accuracy": soft_accuracy
    }, merged

In [ ]:
student_df = pd.read_csv(STUDENT_PREDICTIONS_CSV, encoding="utf-8-sig")
student_df["labels_list"] = student_df.apply(get_labels_list_from_row, axis=1)

student_eval_df = student_df[["id", "labels_list"]].copy()
teacher_eval_df = prepare_eval_df(teacher_path)
manual_eval_df = prepare_eval_df(manual_path)

print("Student:", student_eval_df.shape)
print("Teacher:", teacher_eval_df.shape)
print("Manual:", manual_eval_df.shape)

In [ ]:
teacher_metrics, teacher_compare_df = calculate_metrics_by_labels(base_df=teacher_eval_df, pred_df=student_eval_df)

manual_metrics, manual_compare_df = calculate_metrics_by_labels(base_df=manual_eval_df, pred_df=student_eval_df)

In [ ]:
metrics_df = pd.DataFrame({"Compared with teacher": teacher_metrics, "Compared with manual": manual_metrics})
metrics_df = metrics_df.loc[["Precision", "Recall", "F1", "Exact Match Accuracy", "Soft Accuracy"]]

metrics_df

In [ ]:
with pd.ExcelWriter(METRICS_PATH) as writer:
    metrics_df.to_excel(writer, sheet_name="metrics")
    teacher_compare_df.to_excel(writer, sheet_name="teacher_comparison", index=False)
    manual_compare_df.to_excel(writer, sheet_name="manual_comparison", index=False)

print("Saved metrics locally:", METRICS_PATH)